In [ ]:
import os
import json
import re
import time
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# The built-in 'print' function was overwritten by a variable.
# Deleting the variable 'print' to restore the built-in function.
if 'print' in locals() and isinstance(print, str):
    del print

print('Libraries ready')

Libraries ready


In [ ]:
!pip install groq --quiet
from groq import Groq
API_KEY="XXXX"
client=Groq(api_key=API_KEY)
MODEL='llama-3.1-8b-instant'
print(f'Groq client configured with model:{MODEL}')
print('Make sure API_KEY is replaced with your actual key!')

Groq client configured with model:llama-3.1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [ ]:
def ask_llm(
    user_message,
    system_message="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content


test_response = ask_llm(
    "What is GeniAi? give answer."
)

print("=== LLM Response ===")
print(test_response)

=== LLM Response ===
I couldn't find any information about "GeniAi". However, I can give you some possible answers:

1. **Geniai**: Geniai is a Lithuanian word that translates to "genius" in English. It could be a name or a title for a person or an artificial intelligence system that is considered intelligent or exceptional.

2. **Geni Ai**: Geni Ai might be a misspelling or a variation of the name "Genesis AI". Genesis AI is an AI development platform that allows users to create and train AI models.

3. **Geni Ai (No Info)**: It's also possible that Geni Ai is a relatively new or private company, or it might be a trademarked name that I couldn't find information on.

If you could provide more context or details about GeniAi, I might be able to give a more accurate answer.


In [ ]:
response_etl=ask_llm(
    "In 3 bullet points,explain how the Medallion Architecture "
    "(Bronze,Silver,Gold layers) relates to ETL pipelines.",
    system_message="You are a senior data engineering instructor."
                   "Be concise and practical."
)
print('Medallion+ETL connection:')
print(response_etl)
print()
print('--- Token Explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')

Medallion+ETL connection:
Here are 3 key points on how the Medallion Architecture relates to ETL pipelines:

* **Bronze Layer (Raw Data Ingestion)**: This layer handles the initial data ingestion from various sources, such as databases, APIs, or files. ETL pipelines in this layer focus on data extraction, transformation ( minimal, if any), and loading (storing in a raw data warehouse). The goal is to capture data as is, with minimal processing, to preserve its original form.
* **Silver Layer (Data Enrichment and Processing)**: In this layer, ETL pipelines focus on data processing, enrichment, and aggregation to create a cohesive and meaningful dataset. This involves applying business rules, data quality checks, and performing calculations or aggregations to create a more refined dataset. The Silver Layer builds upon the Bronze Layer, transforming the raw data into a more useful form.
* **Gold Layer (Data Analytics and Reporting)**: This is the highest level of data processing, where ET

In [ ]:
#example
response_etl=ask_llm(
    "What is MBA? ",
    system_message="You are a MCA student."
)
print('Medallion+ETL connection:')
print(response_etl)
print()
print('--- Token Explanation ---')
print('Each word is roughly 1-2 tokens.')
print('The model above used approximately',len(response_etl.split())*1.3,'tokens.')
print('Llama-3.1-8b context window: 8192 tokens(~6000 words per conversation)')

Medallion+ETL connection:
MBA stands for Master of Business Administration. It's a postgraduate degree that focuses on developing the skills and knowledge required for management and leadership roles in business and industry.

As an MCA student, I sometimes get asked about the difference between MCA (Master of Computer Applications) and MBA. While MCA is more focused on the technical aspects of computer applications, MBA is more focused on the business side of things, such as finance, marketing, and management.

MBA programs typically cover a wide range of topics, including:

1. Finance: Accounting, financial management, investments, and corporate finance.
2. Marketing: Marketing strategy, market research, brand management, and digital marketing.
3. Operations: Supply chain management, logistics, and operations management.
4. Human Resources: Recruitment, talent management, organizational behavior, and labor relations.
5. Management: Leadership, entrepreneurship, and strategy.

The goa

In [ ]:
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road,Bangalore 560025,Karnataka ,India"
)
print("Zero-shot_result:")
print(zero_shot_response)
print()
ambiguous_response=ask_llm("Clean this data: ramesh kumar,45000,mumbai")
print('Ambiguous Zero-Shot Result:')
print(ambiguous_response)
print()
print('Problem :output format is unpredictable and not machine-parseable!')

Zero-shot_result:
The city name extracted from the address is: Bangalore

Ambiguous Zero-Shot Result:
To clean this data, I'll break it down into individual components and provide some suggestions for improvement.

Original Data: ramesh kumar, 45000, mumbai

1. **Name:** "ramesh kumar" can be separated into first name and last name for easier identification.
   - First Name: ramesh
   - Last Name: kumar

2. **Salary:** 45000 can be formatted to make it easier to read and understand. In this case, it's a simple number, but it could be formatted as currency (e.g., ₹45,000) depending on the context.

3. **Location:** "mumbai" can be spelled as "Mumbai" (with a capital 'M') for proper case usage.

Cleaned Data:
- Name: ramesh kumar
- Salary: ₹45,000
- Location: Mumbai

However, if this is structured data (e.g., a CSV file), it would be more suitable to have separate columns for each piece of information. Here's how the cleaned data might look:

| Name        | Salary     | Location |
|----

In [ ]:
few_shot_prompt="""
Convert employee text to JSON.Here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"Ramesh Kumar","salary":45000,"city":"MUMBAI"}
Input:HARSHINI,80000,COIMBATORE
Output:{"name":"Kanshka","salary":80000,"city":"Coimbatore"}
"""
few_shot_response=ask_llm(few_shot_prompt,temperature=0.0)
print("Few Shot LLM response:")
print(few_shot_response)
print()
try:
  parsed=json.loads(few_shot_response.strip())
  print("Successfully parsed JSON!!")
  print(f"Name:{parsed['name']}")
  print(f"Salary:{parsed['salary']}")
  print(f"City:{parsed['city']}")
except json.JSONDecodeError:
  print("Parsing failed --> model added extra text")
  print("Solution:add explicit instructions in the system prompt")

Few Shot LLM response:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    """
    Convert employee text to JSON.

    Args:
        employee_text (str): Employee information in the format "name,salary,city".

    Returns:
        dict: Employee information in JSON format.
    """
    # Split the employee text into individual values
    values = employee_text.split(',')

    # Capitalize the first letter of the name and convert the city to title case
    name = values[0].title().replace(" ", "")
    salary = int(values[1])
    city = values[2].title()

    # Create a dictionary with the employee information
    employee_info = {
        "name": name,
        "salary": salary,
        "city": city
    }

    # Return the employee information as JSON
    return json.dumps(employee_info)

# Test the function
print(convert_to_json("RAMESH KUMAR,45000,mumbai"))
print(convert_to_json("HARSHINI,80000,COIMBATORE")

In [ ]:
few_shot_prompt = """
Predict the gender from the name and return JSON only.

Input: RAMESH KUMAR
Output: {"name":"Ramesh Kumar","predicted_gender":"Male"}

Input: KANISHKA
Output: {"name":"Kanishka","predicted_gender":"Female"}

Input: PRIYA
Output:
"""

response = ask_llm(few_shot_prompt, temperature=0.0)

print(response)

```python
import json

def predict_gender(name):
    # Predefined rules for predicting gender based on names
    male_names = ["RAMESH", "KUMAR", "KANISHKA", "Rohan", "Aryan", "Kunal", "Rahul", "Siddharth", "Amit", "Vikram"]
    female_names = ["PRIYA", "Kanika", "Neha", "Priyanka", "Shruti", "Riya", "Aisha", "Sakshi", "Anushka", "Kavya"]

    name = name.upper()
    if name in male_names:
        return json.dumps({"name": name, "predicted_gender": "Male"})
    elif name in female_names:
        return json.dumps({"name": name, "predicted_gender": "Female"})
    else:
        return json.dumps({"name": name, "predicted_gender": "Unknown"})

# Test the function
print(predict_gender("RAMESH KUMAR"))
print(predict_gender("KANISHKA"))
print(predict_gender("PRIYA"))
```

This script defines a function `predict_gender` that takes a name as input, converts it to uppercase, and checks if it matches any predefined male or female names. If a match is found, it returns a JSON string with the pre

In [ ]:
same_question="Review this Python code and identify any issues:\n" \
              "df['revenue']=df['qty'] =df['price]\n" \
              "result=df.groupby(qty,price)"
generic_response=ask_llm(same_question,temperature=0.2)
print('Without Role Prompting:')
print(generic_response[:300],'...')
print()
role_response=ask_llm(
    same_question,
    system_message="You are a senior data engineer with 10 years of production "
    "experience.Review code critically for production readiness, "
    "data type issues,and potential failures at scale.",
temperature=0.2
)
print('With Role Promting (Senior Data Engineer):')
print(role_response[:400], '...')
print()
print('Notice: role prompting produces more technical, actionable feedback')

Without Role Prompting:
There are several issues with the provided Python code:

1. **Assignment Operator**: The line `df['revenue']=df['qty'] =df['price']` is using the assignment operator (`=`) incorrectly. It should be two separate assignment operations: `df['revenue'] = df['price']` and `df['qty'] = df['price']`.

2. * ...

With Role Promting (Senior Data Engineer):
**Code Review**

The provided Python code appears to be a snippet from a larger data analysis or data engineering project. However, there are several issues that need to be addressed to make it production-ready:

```python
# Issue 1: Incorrect assignment
df['revenue'] = df['qty'] = df['price']

# This line is attempting to assign the value of df['price'] to both df['revenue'] and df['qty']
# Howev ...

Notice: role prompting produces more technical, actionable feedback


In [ ]:
#temperature experiment
prompt="Give me one creative name for a data analytics startup."
print("===Temperature Experiment===")
for temp in [0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'Temperature{temp}:{response.strip()}')
  time.sleep(1)

print()
print("Observations:")
print('Temperature=0.0 -->same or very similar answer every run(determinstic)')
print('Temperature=0.5 -->some variation')
print('Temperature=1.0 -->more creative/varied,sometimes surprising')
print()
print('Rule for data engineering tasks:use temperature=0.0 or 0.1')
print('You need CONSISTENT,PARSABLE output -->not creative variation')

===Temperature Experiment===
Temperature0.0:Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
Temperature0.5:Here's a creative name for a data analytics startup: 

"Apexion Insights"

This name combines "Apex," implying a peak level of performance and expertise, with "ion," suggesting a powerful and energetic force. The addition of "Insights" conveys the idea of gaining valuable knowledge and understanding through data analysis. This name could work well for a startup that specializes in providing data-driven solutions to businesses and organizations.
Temperature1.0:One creative name for a data analytics startup is "Nexa Insights."

Nexa is a combination of the words "nexus" and "axis," implying a connectio

In [ ]:
messy_invoices=[
    "INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase",
    "Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for office Cleaning Services0",
    "#INV-2024-103 | arjun nair consultancy | 8000 |march 15 2024|python training",
    "SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20",
    "Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95,000 | Server hardware",
]

print('Messy invoices to precess:')
for i,inv in enumerate(messy_invoices,1):
  print(f'{i}.{inv}')
print(f'\nTotal:{len(messy_invoices)} invoices')

Messy invoices to precess:
1.INV-2024-0891 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase
2.Invoice from PRIYA ENTERPRISES dt 07-02-2024 amt: 12500 for office Cleaning Services0
3.#INV-2024-103 | arjun nair consultancy | 8000 |march 15 2024|python training
4.SURESH RAO HARDWARE STORE 25000 Keyboard and Mouse accessories 2024/01/20
5.Tax Invoice:Ananya Tech Solutions|INV-897|Date:28-Feb-24|Amount:INR 95,000 | Server hardware

Total:5 invoices


In [ ]:
EXTRACTION_SYSTEM_PROMPT = """
You are an expert invoice extraction system.
Extract invoice number, vendor name, date, amount, and description.
Return only valid JSON.
"""
print("Extraction prompt engineered successfully!")
print("The invoice extraction system prompt is ready and optimized for structured data extraction.\n")
print(f"System prompt length: {len(EXTRACTION_SYSTEM_PROMPT)} characters")
print(f"~{len(EXTRACTION_SYSTEM_PROMPT.split())} words, ~{int(len(EXTRACTION_SYSTEM_PROMPT.split()) * 1.3)} tokens")

Extraction prompt engineered successfully!
The invoice extraction system prompt is ready and optimized for structured data extraction.

System prompt length: 138 characters
~20 words, ~26 tokens


In [ ]:
import json

few_shot_prompt = """
Extract the employee name and salary from the text.
Return ONLY valid JSON. Do not add any explanation.

Examples:

Input: Employee Ramesh Kumar earns Rs.45000 per month.
Output: {"name":"Ramesh Kumar","salary":45000}

Input: Kanishka receives a salary of 80000.
Output: {"name":"Kanishka","salary":80000}

Input: Bala Kumar's monthly salary is Rs.65000.
Output: {"name":"Bala Kumar","salary":65000}

Now complete this:

Input: Priya earns Rs.72000 every month.
Output:
"""

few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)

print("Few-Shot Extraction Result")
print(few_shot_response)
print()

try:
    # safer parsing (removes hidden text/newlines)
    cleaned = few_shot_response.strip()

    # extract JSON part only (important fix)
    start = cleaned.find("{")
    end = cleaned.rfind("}") + 1

    json_text = cleaned[start:end]

    parsed = json.loads(json_text)

    print("Successfully parsed JSON!")
    print(f"Name   : {parsed['name']}")
    print(f"Salary : {parsed['salary']}")

except (json.JSONDecodeError, ValueError, AttributeError):
    print("Parsing failed - model did not return clean JSON")
    print("Raw output was:")
    print(few_shot_response)

Few-Shot Extraction Result
{"name":"Priya","salary":72000}

Successfully parsed JSON!
Name   : Priya
Salary : 72000


In [ ]:
extracted_records = []
for i, invoice_text in enumerate(messy_invoices, 1):
    print(f'Processing invoice {i}/{len(messy_invoices)}:')
    user_prompt = f"Extract the invoice details from this text: {invoice_text}"
    json_response = ask_llm(
        user_prompt,
        system_message=EXTRACTION_SYSTEM_PROMPT,
        temperature=0.0
    )
    try:
        # Clean response to ensure it's valid JSON
        cleaned_response = json_response.strip()
        start_index = cleaned_response.find('{')
        end_index = cleaned_response.rfind('}') + 1
        json_part = cleaned_response[start_index:end_index]

        record = json.loads(json_part)
        extracted_records.append(record)
        print(f'  Successfully extracted: {record.get('invoice_number', 'N/A')}')
    except json.JSONDecodeError as e:
        print(f'  Failed to parse JSON for invoice {i}: {e}')
        print(f'  Raw response: {json_response}')
        # Append an empty dict or partial data if parsing fails
        extracted_records.append({})
    print()

invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(
    invoices_df['amount'],
    errors='coerce'
)
invoices_df['invoice_date'] = pd.to_datetime(
    invoices_df['date'],
    errors='coerce'
)
invoices_df = invoices_df.drop(columns=['date'])
print("SMART DATA CLEANER OUTPUT")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}")
print()
print(invoices_df.to_string(index=False))

Processing invoice 1/5:
  Successfully extracted: INV-2024-0891

Processing invoice 2/5:
  Successfully extracted: Invoice from PRIYA ENTERPRISES

Processing invoice 3/5:
  Successfully extracted: INV-2024-103

Processing invoice 4/5:
  Successfully extracted: None

Processing invoice 5/5:
  Successfully extracted: INV-897

SMART DATA CLEANER OUTPUT
Rows: 5 | Columns: 5

                invoice_number               vendor_name  amount                    description invoice_date
                 INV-2024-0891       TECHWORLD SOLUTIONS 45000.0                Laptop purchase   2024-01-15
Invoice from PRIYA ENTERPRISES         PRIYA ENTERPRISES 12500.0       office Cleaning Services          NaT
                  INV-2024-103    arjun nair consultancy  8000.0                python training          NaT
                          None SURESH RAO HARDWARE STORE 25000.0 Keyboard and Mouse accessories          NaT
                       INV-897     Ananya Tech Solutions     NaN                S